In [32]:
import re
import spikeinterface.extractors as se
import spikeinterface as si

import numpy as np
from pipeline.sync import get_kilosort_spikes, match_chirp_edges
from pipeline.utils import log_recording, load_events, parse_timestamps, log_timestamps
from pathlib import Path
from loguru import logger
from scipy.interpolate import make_interp_spline

class Timestamps:


    def __init__(self, name: str, fs: float, t_start: float = 0.0):
        self.name = name
        self.fs = fs
        self.dt = 1 / fs
        
        self.global_timestamps = []
        self.intervals = []
        self.t_offset = t_start
        self.starting_states = []

    def update(self, local_ts: np.ndarray, t_end: float, starting_state = None) -> None:
        """Add new session to global timeline."""
        # Update global timestamps
        self.global_timestamps.append(local_ts + self.t_offset + self.dt)

        # Update intervals
        self.intervals.append((self.t_offset, self.t_offset + t_end))
        
        # Update offset for next segment
        self.t_offset += t_end

        if starting_state:
            self.starting_states.append(starting_state)
        
    def __repr__(self):
        return f"Timestamps(name='{self.name}', @ {self.fs:.1f} Hz)"


class MultiProbeRun:
    """ Represents multiple probes in a single OpenEphys session ."""


    def __init__(self, run_path):
        logger.info(f"Initializing MultiProbeRun for path: {run_path}")
        self.run_path = run_path
        self.session_name = Path(run_path).name
        self._probes = {}

    def load_recordings(self):
        """ Load recordings for all probes matching the filter."""
        logger.info(f"Loading recordings from {self.run_path}")
        stream_names, stream_ids = se.get_neo_streams('openephysbinary', self.run_path)

        for stream_name, stream_id in zip(stream_names, stream_ids):
            logger.info(f"Found stream: {stream_name} with ID: {stream_id}")
            # Extract probe name (e.g., "OneBox-0.ProbeA" -> "ProbeA")
            probe_name = stream_name.split(".")[-1]
            if "SYNC" not in probe_name:
                logger.info(f"Loading probe: {probe_name}")
                pr = se.read_openephys(self.run_path, stream_id=stream_id)
                log_recording(pr, f"Loaded_Probe_{probe_name}")
                logger.debug(f"File size: {pr.get_total_memory_size()}%{pr.get_num_channels()}=={pr.get_total_memory_size()%pr.get_num_channels()}")
                self._probes[probe_name] = pr

    def __getitem__(self, probe_name):
        return self._probes[probe_name]

    def __repr__(self):
        return f"MultiProbeRun(run_path={self.run_path}, probes={list(self._probes.keys())})"

class Experiment:
    """ Represents an experiment with multiple sessions."""


    def __init__(self, recording_paths):
        self.recording_paths = recording_paths
        self.sessions = [MultiProbeRun(path) for path in recording_paths]
        self.sync_adc = Timestamps(name='OneBox-ADC', fs=30300.5, t_start=0.0)
        self.sync_probes = {}
        self._probes = {}

    def preprocess(self):
        """ Preprocess all sessions."""
        for session in self.sessions:
            logger.info(f"Preprocessing session at {session.run_path}")
            session.load_recordings()

    def concatenate(self, probe_filter, save_path=None):
        """ Concatenate recordings for each probe across sessions."""
        for probe_name in probe_filter:
            logger.info(f"Concatenating sessions for probe: {probe_name}")
            probe_recordings = [session[probe_name] for session in self.sessions]
            concatenated_recording = si.concatenate_recordings(probe_recordings)
            log_recording(concatenated_recording, f"Concatenated_Probe_{probe_name}")
            self._probes[probe_name] = concatenated_recording
        
        if save_path:
            for probe_name, recording in self._probes.items():
                recording.save(save_path / f"Probe_{probe_name}")
        return self._probes

    def sync(self, probe_filter, kilosort_path):
        """ Sync all probes to ADC timeline."""
        logger.info("RUNNING SYNCHRONIZATION")
        timestamps = parse_timestamps(self.recording_paths)
        self._load_adc(timestamps['OneBox-ADC'])

        # Load Kilosort spike times
        logger.info("Loading Kilosort spike times")
        ks_spikes = get_kilosort_spikes(output_path=kilosort_path, probe_filter=probe_filter)

        logger.info("Interpolating spikes to ADC global timebase")
        self.sync_probes = {k:d for k,d in timestamps.items() if k != "OneBox-ADC" and k in probe_filter}

        for probe, paths in self.sync_probes.items():
            PRB = Timestamps(name=probe, fs=30000.0, t_start=0.0)
            adc_global_timestamps = []
            logger.info(f"Processing probe: {probe}")

            logger.info("Extracting kilosort spikes")
            kilosort_spikes = ks_spikes[probe] / PRB.fs
            log_timestamps(kilosort_spikes, f"{probe} spikes")
            total_spikes_left = kilosort_spikes.size
            logger.info('='*60)

            save_dir = Path(kilosort_path / probe)
            save_dir.mkdir(parents=True, exist_ok=True)
            logger.info(f"Saving to: {save_dir}")

            synced_spikes   = []

            for idx, (ev_path, cont_path) in enumerate(zip(paths['event'], paths['cont'])):
                event_ts, cont_ts, states = load_events(ev_path, cont_path)

                # Handle state mismatches
                if states[0] != self.sync_adc.starting_states[idx]:
                    logger.warning(f"State mismatch between {probe} and ADC")
                    logger.info("Matching edges")
                    event_ts, _ = match_chirp_edges(event_ts, self.sync_adc.global_timestamps[idx])

                # Update probe timestamps, starting state is not needed here
                PRB.update(event_ts - cont_ts[0], cont_ts[-1] - cont_ts[0])

                probe_times = PRB.global_timestamps[-1]
                ########## LOGGING #################################
                log_timestamps(event_ts, f"Event timestamps")
                log_timestamps(cont_ts, f"Continuous timestamps")
                log_timestamps(probe_times, f"Global segment")
                log_timestamps(np.concatenate(PRB.global_timestamps), f"Global")
                ########## LOGGING #################################

                adc_times = self.sync_adc.global_timestamps[idx]
                
                # Extract spikes based on continuous range
                cont_start, cont_end = PRB.intervals[idx]
                logger.info(f"Extracting spikes in interval: {cont_start:.5f} ... {cont_end:.5f} s")
                mask = (kilosort_spikes > cont_start) & (kilosort_spikes <= cont_end)
                # masks.append(mask) # DEBUG purpose
                
                probe_spikes = kilosort_spikes[mask]
                log_timestamps(probe_spikes, f"Extracted spikes")
                
                # Handle length mismatches
                min_length = min(len(probe_times), len(adc_times))
                if min_length < len(adc_times):
                    logger.warning(f"  Truncating ADC timestamps. ADC timestamps: {len(adc_times)} -> {min_length}.")
                    adc_times = adc_times[:min_length]
                elif min_length < len(probe_times):
                    logger.warning(f"  Truncating Probe timestamps. Probe timestamps: {len(probe_times)} -> {min_length}.")
                    PRB.global_timestamps[idx] = probe_times[:min_length]
                
                adc_global_timestamps.append(adc_times)

                # Interpolate/extrapolate to ADC time
                spl = make_interp_spline(x=probe_times, y=adc_times, k=1)
                adc_spikes = spl(probe_spikes)
                synced_spikes.append(adc_spikes)
                total_spikes_left -= adc_spikes.size

                log_timestamps(adc_spikes, "ADC interpolated spikes")
                logger.info(f"Synced spikes: {adc_spikes.size}/{kilosort_spikes.size}. Remaining spikes: {total_spikes_left}")
                logger.info("-"*60)
            
            # Create and save timestamps map Probe Global timestamps <-> ADC Global timestamps
            probe_times = np.concatenate(PRB.global_timestamps)
            adc_times = np.concatenate(adc_global_timestamps)
            timestamps_map = np.vstack((probe_times, adc_times)).T

            np.save(save_dir / "timestamps_map.npy", timestamps_map)
            np.save(save_dir / "adc_spikes.npy", np.concatenate(synced_spikes))
            
            # np.save(save_dir / "masks.npy", np.concatenate(masks))
            # np.save(save_dir / "intervals.npy", PRB.intervals)
            
            logger.success(f"Completed synchronization for probe: {probe}")
        logger.success("SYNCHRONIZATION COMPLETED")

        


    def _load_adc(self, timestamps):
        """ Compute global timestamps."""
        adc_event_paths, adc_cont_paths = timestamps['event'], timestamps['cont']

        for idx, (event_path, cont_path) in enumerate(zip(adc_event_paths, adc_cont_paths)):
            event_ts, cont_ts, states = load_events(event_path, cont_path)
            # Subtract the offset of continuous recording
            self.sync_adc.update(event_ts - cont_ts[0], cont_ts[-1] - cont_ts[0], starting_state=states[0])

            ########## LOGGING #################################
            log_timestamps(event_ts, "ADC event")
            log_timestamps(cont_ts, "ADC cont")
            logger.info(f"ADC interval: {self.sync_adc.intervals[-1][0]:.4f} ... {self.sync_adc.intervals[-1][1]:.4f} s")
            log_timestamps(self.sync_adc.global_timestamps[-1], "ADC global segment")
            log_timestamps(np.concatenate(self.sync_adc.global_timestamps), "ADC global")
            logger.info("-"*60)
            ####################################################


    def __repr__(self):
        return f"Experiment(sessions={len(self.sessions)})"


In [3]:
from pipeline.utils import load_config
conf_path = Path('/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/pipeline_output/configs/config_example.yaml')
config = load_config(conf_path)
config

2025-11-10 12:04:37.981 | SUCCESS  | pipeline.utils:load_config:66 - Loaded configuration from: /Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/pipeline_output/configs/config_example.yaml


{'session_name': 'AA001_Day2',
 'recording_paths': ['R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\VisualStimuli\\AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual',
  'R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField',
  'R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_14-22-26_4Probe_RSC_ADn_RecOpenField',
  'R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_14-57-57_4Probe_RSC_ADn_RecOpenField'],
 'local_output': 'E:/pipeline_output',
 'remote_output': 'R:/Basic_Sciences/Phys/SenzaiLab/pipeline_output',
 'fs': 30000.0,
 'target_fs': 1250.0,
 'save_kwargs': {'n_jobs': 16,
  'chunk_duration': '2s',
  'progress_bar': True,
  'mp_context': 'spawn',
  'verbose': True,
  'overwrite': True}}

In [7]:
recording_paths = ['/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual',
  '/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField',
  '/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_14-22-26_4Probe_RSC_ADn_RecOpenField',
  '/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_14-57-57_4Probe_RSC_ADn_RecOpenField']

In [28]:
session1 = MultiProbeRun(recording_paths[0])
session1.load_recordings()

2025-11-10 12:31:46.480 | INFO     | __main__:__init__:45 - Initializing MultiProbeRun for path: /Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual
2025-11-10 12:31:46.480 | INFO     | __main__:load_recordings:52 - Loading recordings from /Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual
2025-11-10 12:31:47.943 | INFO     | __main__:load_recordings:56 - Found stream: Record Node 103#OneBox-106.OneBox-ADC with ID: 0
2025-11-10 12:31:47.944 | INFO     | __main__:load_recordings:60 - Loading probe: OneBox-ADC
2025-11-10 12:31:49.021 | INFO     | pipeline.utils:log_recording:179 - Loaded_Probe_OneBox-ADC: 12 ch, 3634.1s (1.01 h) @ 30.3 kHz int16 (2.46 GB)
2025-11-10 12:31:49.022 | DEBUG    | pipeline.utils:log_recording:183 - Filepath: /Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-

In [33]:
exp = Experiment(recording_paths)
exp.preprocess()

2025-11-10 14:53:47.254 | INFO     | __main__:__init__:48 - Initializing MultiProbeRun for path: /Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual
2025-11-10 14:53:47.254 | INFO     | __main__:__init__:48 - Initializing MultiProbeRun for path: /Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField
2025-11-10 14:53:47.255 | INFO     | __main__:__init__:48 - Initializing MultiProbeRun for path: /Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_14-22-26_4Probe_RSC_ADn_RecOpenField
2025-11-10 14:53:47.255 | INFO     | __main__:__init__:48 - Initializing MultiProbeRun for path: /Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_14-57-57_4Probe_RSC_ADn_RecOpenField
2025-11-10 14:53:47.923 | INFO     | __main__:preprocess:89 - Prepr

In [ ]:
exp.sync(probe_filter=['ProbeA'], kilosort_path=Path(config['kilosort_output_path']))

2025-11-10 12:11:43.285 | INFO     | __main__:concatenate:85 - Concatenating sessions for probe: ProbeA
2025-11-10 12:11:43.286 | INFO     | pipeline.utils:log_recording:179 - Concatenated_Probe_ProbeA: 384 ch, 33692.2s (9.36 h) @ 30.0 kHz int16 (722.96 GB)
2025-11-10 12:11:43.287 | INFO     | __main__:concatenate:85 - Concatenating sessions for probe: ProbeB
2025-11-10 12:11:43.287 | INFO     | pipeline.utils:log_recording:179 - Concatenated_Probe_ProbeB: 384 ch, 34245.8s (9.51 h) @ 30.0 kHz int16 (734.83 GB)


{'ProbeA': ConcatenateSegmentRecording: 384 channels - 30.0kHz - 1 segments - 1,010,766,713 samples 
                              33,692.22s (9.36 hours) - int16 dtype - 722.96 GiB,
 'ProbeB': ConcatenateSegmentRecording: 384 channels - 30.0kHz - 1 segments - 1,027,373,501 samples 
                              34,245.78s (9.51 hours) - int16 dtype - 734.83 GiB}

In [12]:
exp._probes['ProbeA'].get_total_memory_size()

776268835584

In [13]:
from pipeline.utils import format_file_size
format_file_size(exp._probes['ProbeA'].get_total_memory_size())

'722.96 GB'

In [18]:
list(Path(session1.run_path).rglob('timestamps.npy'))

[PosixPath('/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual/Record Node 103/experiment1/recording1/events/MessageCenter/timestamps.npy'),
 PosixPath('/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual/Record Node 103/experiment1/recording1/events/OneBox-106.ProbeC/TTL/timestamps.npy'),
 PosixPath('/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual/Record Node 103/experiment1/recording1/events/OneBox-106.ProbeD/TTL/timestamps.npy'),
 PosixPath('/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual/Record Node 103/experiment1/recording1/events/OneBox-106.OneBox-ADC/TTL/timestamps.npy'),
 PosixPath('/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_20